# 📝 그래프DB 개념 과제 LV1 정답: SNS 팔로우 그래프 (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../data/` 입니다.

- 채점은 고정 데이터(`sns_follows.json`) 위의 **결정적 파이썬**이라 매번 같은 결과입니다.
- 그래프는 순수 파이썬 dict/list 로 표현합니다(노드 dict + 관계 list).

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 채점에 쓰는 그래프 데이터를 먼저 훑어봅니다.

In [ ]:
# [제공 코드] SNS 팔로우 그래프를 불러옵니다
import json
from pathlib import Path
graph = json.loads(Path('../data/sns_follows.json').read_text(encoding='utf-8'))
nodes = graph['nodes']      # id -> {'name': 이름, 'type': 사용자/브랜드, 'region': 지역}
edges = graph['edges']      # [{'relation': 관계, 'source': 출발 id, 'target': 도착 id}, ...]
print('노드', len(nodes), '개 · 관계', len(edges), '개')
print('노드 예:', list(nodes.items())[0])
for e in edges[:3]:
    print(e)

## 1. 팔로잉 수·팔로워 수(차수) 세기
**배경**: 노드에서 **나가는** 화살표 수와 **들어오는** 화살표 수를 그 노드의 **차수**라고 합니다. 팔로우 그래프에서는 각각 팔로잉 수와 팔로워 수가 됩니다.

**요구사항**:
- **`팔로우`** 관계만 써서 두 사전을 만드세요(관심 관계는 세지 않습니다).
  - **`out_deg`**: 사용자 id -> 그 사용자가 팔로우하는 사람 수
  - **`in_deg`**: 사용자 id -> 그 사용자를 팔로우하는 사람 수

**예시**: `out_deg['u1']` 은 **3**, `in_deg['u1']` 은 **2** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 팔로우 엣지 하나가 출발 노드의 out 을 1, 도착 노드의 in 을 1 올린다.

세부구현:
1. 빈 dict 두 개를 만든다.
2. edges 를 돌며 relation 이 '팔로우' 인 엣지만 본다.
3. source 를 키로 out_deg 값을 1 더하고, target 을 키로 in_deg 값을 1 더한다.
```

</details>

In [ ]:
# 엣지 하나 = 출발 노드의 나가는 차수 1, 도착 노드의 들어오는 차수 1
out_deg = {}
in_deg = {}
for e in edges:
    if e['relation'] == '팔로우':
        out_deg[e['source']] = out_deg.get(e['source'], 0) + 1
        in_deg[e['target']] = in_deg.get(e['target'], 0) + 1
print('팔로잉 수:', out_deg)
print('팔로워 수:', in_deg)

In [ ]:
# [자가채점]
assert out_deg['u1'] == 3 and out_deg['u5'] == 1, \
    'out_deg 는 source 를 키로 세야 합니다(target 으로 세면 팔로워 수가 됩니다)'
assert in_deg['u1'] == 2 and in_deg['u7'] == 2, \
    'in_deg 는 target 을 키로 세야 합니다(source 로 세면 방향이 뒤집힙니다)'
assert len(out_deg) == 8 and len(in_deg) == 8, \
    '팔로우 관계만 세었는지 확인하세요(관심 관계까지 세면 브랜드 노드가 섞입니다)'
assert max(in_deg.values()) == 2, '팔로워가 가장 많은 사람도 2명입니다. 값을 다시 세어 보세요'
assert len([u for u, d in in_deg.items() if d == 2]) == 5, \
    '팔로워 2명인 사용자는 5명이어야 합니다'
print('✅ 통과!')

**해설**: 엣지를 한 번만 훑으면 두 차수를 동시에 셀 수 있습니다. `관심` 관계까지 세면 브랜드 노드가 `in_deg` 에 섞여 들어와 사전 크기가 10이 되므로, **관계 종류를 먼저 거르는 것**이 이 문제의 핵심입니다.

**흔한 실수**: 1) 관계 종류를 안 걸러 브랜드가 섞이는 경우. 2) `in_deg` 를 만들 때 `source` 를 키로 쓰는 경우(방향이 뒤집힙니다).

## 2. 팔로우 관계 개수 세기
**배경**: 관계에는 `팔로우` 와 `관심` 두 종류가 섞여 있습니다. 그중 **팔로우** 관계만 셉니다.

**요구사항**:
- `edges` 에서 `relation` 이 **`'팔로우'`** 인 것만 골라 리스트 **`follow_edges`** 에 담고, 그 개수를 **`n_follow`** 에 담으세요.

**예시**: 팔로우 관계는 **13개** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 리스트 컴프리헨션으로 relation 이 '팔로우' 인 엣지만 남긴다.

세부구현:
1. 리스트 컴프리헨션으로 relation 이 '팔로우' 인 엣지만 남겨 follow_edges 에 담는다.
2. follow_edges 의 길이를 n_follow 에 담는다.
```

</details>

In [ ]:
# 관계가 두 종류 섞여 있으므로 세기 전에 종류부터 거른다
follow_edges = [e for e in edges if e['relation'] == '팔로우']
n_follow = len(follow_edges)
print('팔로우 관계:', n_follow)

In [ ]:
# [자가채점]
assert n_follow == len(follow_edges) == 13, \
    'n_follow 는 follow_edges 의 길이여야 합니다(팔로우 엣지 13개)'
assert all(e['relation'] == '팔로우' for e in follow_edges), \
    'follow_edges 에 관심 관계가 섞였습니다. relation 이 팔로우 인 것만 남기세요'
print('✅ 통과!')

**해설**: `relation` 값으로 거르는 기본 필터입니다. 1번이 전체 세기라면 여기선 조건에 맞는 것만 셉니다.

## 3. 내가 팔로우하는 사람(팔로잉) 조회
**배경**: 팔로우 관계는 **방향**이 있습니다. `민준(u1)` 이 **팔로우하는** 사람은 화살표가 `u1 → ?` 로 나가는 쪽입니다.

**요구사항**:
- 팔로우 관계 중 `source` 가 **`'u1'`** 인 엣지의 `target` 을 모아 **집합(set)** **`following_u1`** 에 담으세요.

**예시**: `following_u1` 의 크기는 **3** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 팔로우 엣지 중 source 가 'u1' 인 것의 target 을 set 으로 모은다.

세부구현:
1. edges 를 돌며 relation=='팔로우' 이고 source=='u1' 인 엣지를 찾는다.
2. 그 엣지들의 target 을 집합(set)으로 모아 following_u1 에 담는다.
```

</details>

In [ ]:
# 내가 팔로우하는 사람 = 화살표가 나에게서 '나가는' 쪽이므로 source 를 고정하고 target 을 모은다
following_u1 = {e['target'] for e in edges if e['relation'] == '팔로우' and e['source'] == 'u1'}
print('민준이 팔로우:', following_u1)

In [ ]:
# [자가채점]
assert following_u1 == {'u2', 'u3', 'u4'}, \
    'u1 이 source 인 팔로우 엣지의 target 을 모았는지 확인하세요(방향이 반대면 팔로워가 됩니다)'
print('✅ 통과!')

**해설**: 방향이 있는 관계라 `source` 가 나(u1)인 엣지의 `target` 이 "내가 팔로우하는 사람"입니다.

## 4. 나를 팔로우하는 사람(팔로워) 조회
**배경**: 이번엔 방향을 **거꾸로** 봅니다. `민준(u1)` 을 팔로우하는 사람은 화살표가 `? → u1` 로 들어오는 쪽입니다.

**요구사항**:
- 팔로우 관계 중 `target` 이 **`'u1'`** 인 엣지의 `source` 를 모아 집합 **`followers_u1`** 에 담으세요.

**예시**: `followers_u1` 의 크기는 **2** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3번과 반대로, target 이 'u1' 인 엣지의 source 를 모은다.

세부구현:
1. edges 를 돌며 relation=='팔로우' 이고 target=='u1' 인 엣지를 찾는다.
2. 그 엣지들의 source 를 집합으로 모아 followers_u1 에 담는다.
```

</details>

In [ ]:
# 팔로워는 방향이 반대 - target 을 고정하고 source 를 모은다
followers_u1 = {e['source'] for e in edges if e['relation'] == '팔로우' and e['target'] == 'u1'}
print('민준의 팔로워:', followers_u1)

In [ ]:
# [자가채점]
assert followers_u1 == {'u7', 'u8'}, \
    'u1 이 target 인 팔로우 엣지의 source 를 모았는지 확인하세요(3번과 방향이 반대입니다)'
print('✅ 통과!')

**해설**: 팔로잉(3번)과 팔로워(4번)는 **방향만 반대**입니다. `source`↔`target` 를 바꾸면 됩니다.

## 5. 속성으로 필터: 서울에 사는 사용자
**배경**: 노드에는 `region`(지역) 속성이 있습니다. 속성으로 노드를 걸러 봅니다.

**요구사항**:
- `nodes` 에서 `type` 이 **`'사용자'`** 이고 `region` 이 **`'서울'`** 인 노드의 **id** 를 모아 집합 **`seoul_users`** 에 담으세요.

**예시**: 서울 사용자는 **4명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- nodes.items() 를 돌며 두 조건(type·region)을 모두 만족하는 id 를 모은다.

세부구현:
1. for nid, a in nodes.items() 로 돈다.
2. a['type']=='사용자' 이고 a['region']=='서울' 이면 nid 를 집합에 넣는다.
```

</details>

In [ ]:
# 두 조건을 모두 만족해야 하므로 and 로 묶는다(브랜드도 region 이 있어 type 검사가 필요)
seoul_users = {nid for nid, a in nodes.items() if a['type'] == '사용자' and a['region'] == '서울'}
print('서울 사용자:', seoul_users)

In [ ]:
# [자가채점]
assert seoul_users == {'u1', 'u2', 'u4', 'u7'}, \
    'type 과 region 두 조건을 and 로 함께 봐야 합니다(브랜드에도 region 이 있습니다)'
print('✅ 통과!')

**해설**: 두 조건을 `and` 로 묶어 거릅니다. 노드의 속성은 `a['키']` 로 읽습니다.

## 6. 레이블(type)별 노드 세기
**배경**: 노드의 `type` 은 레이블(사용자/브랜드) 역할을 합니다. 레이블별로 몇 개인지 셉니다.

**요구사항**:
- `nodes` 의 각 노드 `type` 별 개수를 사전 **`by_type`** 에 담으세요(키=type, 값=개수).

**예시**: `by_type['사용자']` 는 **8**, `by_type['브랜드']` 는 **2** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- collections.Counter 로 각 노드의 type 값을 센다.

세부구현:
1. collections 에서 Counter 를 불러온다.
2. Counter 로 각 노드의 type 값을 세어 dict 로 만들어 by_type 에 담는다.
```

</details>

In [ ]:
# 레이블별 빈도는 직접 세는 대신 Counter 에 맡긴다
from collections import Counter
by_type = dict(Counter(a['type'] for a in nodes.values()))
print(by_type)

In [ ]:
# [자가채점]
assert by_type['사용자'] == 8, '노드의 type 값을 세었는지 확인하세요(엣지가 아니라 노드입니다)'
assert by_type['브랜드'] == 2, '브랜드 노드도 빠짐없이 세었는지 확인하세요'
print('✅ 통과!')

**해설**: `Counter` 가 레이블별 빈도를 한 번에 셉니다. `nodes.values()` 로 속성만 꺼내 `type` 을 봅니다.

## 7. 관계를 트리플로 변환
**배경**: 그래프의 관계 `{relation, source, target}` 를 지식 그래프의 **트리플 `(주어, 관계, 목적어)`** 로 옮겨 봅니다. 이때 id 대신 **이름**을 씁니다.

**요구사항**:
- 함수 **`edge_to_triple(e)`** 를 만드세요. 엣지 dict `e` 를 받아 `(출발 이름, 관계, 도착 이름)` **튜플**을 돌려줍니다. 이름은 `nodes[id]['name']` 으로 얻습니다.

**예시**: `edge_to_triple(edges[0])` 는 `('민준', '팔로우', '서연')`, `edge_to_triple(edges[13])` 는 `('민준', '관심', '데일리카페')` 입니다(관계 종류가 달라도 같은 규칙으로 바꿉니다).

<details><summary>힌트</summary>

```text
접근방법:
- e 에서 source·relation·target 을 꺼내고, id 를 nodes[id]['name'] 으로 이름으로 바꾼다.

세부구현:
1. 출발 이름 = nodes[e['source']]['name'], 도착 이름 = nodes[e['target']]['name']
2. (출발 이름, e['relation'], 도착 이름) 튜플을 반환한다.
```

</details>

In [ ]:
# id 는 기계용 키일 뿐 - 사람이 읽는 트리플로 만들려면 이름으로 바꿔 준다
def edge_to_triple(e):
    """엣지 e 를 (출발 이름, 관계, 도착 이름) 튜플로 바꿔 돌려준다."""
    return (nodes[e['source']]['name'], e['relation'], nodes[e['target']]['name'])

print(edge_to_triple(edges[0]))

In [ ]:
# [자가채점]
assert edge_to_triple(edges[0]) == ('민준', '팔로우', '서연'), \
    'id 를 nodes[id][\'name\'] 으로 바꿔 (출발 이름, 관계, 도착 이름) 튜플을 돌려주세요'
assert edge_to_triple(edges[13]) == ('민준', '관심', '데일리카페'), \
    '관계 종류가 다른 엣지도 같은 규칙으로 변환해야 합니다(값을 고정해 두지 마세요)'
print('✅ 통과!')

**해설**: 엣지의 id 를 이름으로 바꾸면 사람이 읽는 트리플이 됩니다. 방향(source→target)이 곧 주어→목적어입니다.

## 8. 2홉 도달: 친구의 친구
**배경**: `민준(u1)` 이 팔로우하는 사람(1홉)이 팔로우하는 사람이 **2홉**입니다. 단, 이미 1홉인 사람과 자기 자신은 뺍니다(추천 후보를 새 사람으로만).

**요구사항**:
- 먼저 팔로우의 **인접 dict** `out`(`id → 팔로우하는 id 들의 집합`)을 만드세요.
- `u1` 의 1홉(`out['u1']`)에서 각 사람의 팔로우 대상을 모아 2홉 집합을 만들되, **1홉과 `'u1'` 자신은 제외**하고 **`two_hop`** 에 담으세요.

**예시**: `two_hop` 의 크기는 **4** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 팔로우 엣지로 인접 dict 를 만든 뒤, 1홉 각각의 이웃을 합치고 1홉·자신을 뺀다.

세부구현:
1. 빈 dict out 을 두고, 팔로우 엣지마다 setdefault 로 source 의 집합에 target 을 추가한다.
2. out 에서 'u1' 의 1홉 집합 one 을 얻는다.
3. 빈 집합에 one 의 각 원소가 팔로우하는 이웃 집합을 합집합으로 모은다.
4. 그 결과에서 one 과 'u1' 자신을 빼서 two_hop 에 담는다.
```

</details>

In [ ]:
# 순회를 반복할 것이므로 인접 dict 를 먼저 만들어 두면 이후가 단순해진다
out = {}
for e in edges:
    if e['relation'] == '팔로우':
        out.setdefault(e['source'], set()).add(e['target'])

# 1홉의 이웃을 다시 펼치면 2홉 - 1홉과 자신을 빼야 '새로 알게 되는 사람' 만 남는다
one = out['u1']
two = set()
for m in one:
    two |= out.get(m, set())
two_hop = two - one - {'u1'}
print('2홉 도달:', two_hop)

In [ ]:
# [자가채점]
assert two_hop == {'u5', 'u6', 'u7', 'u8'}, \
    '2홉 결과에서 1홉과 u1 자신을 뺐는지 확인하세요'
print('✅ 통과!')

**해설**: 1홉의 이웃을 다시 한 번 펼치면 2홉입니다. 1홉·자신을 빼야 "새로 알게 되는 사람"만 남습니다(친구 추천의 기본 아이디어).

---
## SPARQL 로 질의하기

여기서부터는 같은 그래프를 **RDF 로 옮겨** 교안에서 배운 SPARQL 문법을 하나씩 써 봅니다. 아래 셀을 **실행만** 하면 `sns_graph` 와 `QUERY_PREFIX` 가 준비됩니다.

이 그래프에는 세 가지가 들어 있습니다.

- 관계: `ex:팔로우`(사용자→사용자), `ex:관심`(사용자→브랜드)
- 타입: `ex:민준 a ex:사용자` 처럼 `rdf:type` 으로 적어 둔 것
- 값: `ex:지역` 의 목적어는 `"서울"` 같은 **리터럴**입니다
- 계층: `ex:사용자 rdfs:subClassOf ex:계정` 처럼 **타입끼리의 상하 관계**도 적혀 있습니다

In [ ]:
# [제공 코드] SNS 그래프를 rdflib 의 RDF 그래프로 옮긴다(실행만 하세요)
from rdflib import Graph, Literal, Namespace, RDF, RDFS

EX = Namespace('http://example.org/sns/')
sns_graph = Graph()
sns_graph.bind('ex', EX)
for node_id, attr in nodes.items():
    # 타입은 표준 술어 rdf:type 으로, 지역은 값(리터럴)으로 담는다
    sns_graph.add((EX[attr['name']], RDF.type, EX[attr['type']]))
    sns_graph.add((EX[attr['name']], EX['지역'], Literal(attr['region'])))
for e in edges:
    sns_graph.add((EX[nodes[e['source']]['name']], EX[e['relation']],
                   EX[nodes[e['target']]['name']]))


# 타입끼리의 계층도 적어 둔다. 사용자도 브랜드도 넓게 보면 '계정' 이다
sns_graph.add((EX['사용자'], RDFS.subClassOf, EX['계정']))
sns_graph.add((EX['브랜드'], RDFS.subClassOf, EX['계정']))

# 질의문에서 실제로 쓰는 이름표만 적어 둔다(rdfs: 는 마지막 계층 문제에서 쓴다)
QUERY_PREFIX = ('PREFIX ex: <http://example.org/sns/>\n'
                'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n')
print('트리플 수:', len(sns_graph))   # 출력: 39

## 9. SPARQL 기본 패턴
**배경**: 3번에서 파이썬으로 구한 "민준이 팔로우하는 사람"을 이번엔 **질의 한 줄**로 구합니다.

**요구사항**:
- 주어를 `ex:민준` 으로 고정하고 술어를 `ex:팔로우`, 목적어를 변수로 둔 패턴 한 줄을 씁니다.
- 결과 값은 URI 이므로 마지막 칸만 잘라 **이름** 집합 **`sparql_following`** 에 담으세요.

**예시**: 민준이 팔로우하는 사람은 **3명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 목적어 자리만 변수로 두면 '민준이 팔로우하는 대상' 이 모인다.

세부구현:
1. QUERY_PREFIX 뒤에 SELECT 와 WHERE 를 이어 붙인다.
2. WHERE 안에 주어·술어를 고정하고 목적어만 변수로 둔 패턴 한 줄을 적는다.
3. sns_graph 에 실행하고, 각 행의 값을 str 로 바꿔 마지막 칸만 잘라 집합에 담는다.
```

</details>

In [ ]:
# 주어와 술어를 고정하고 목적어만 변수로 두면 '민준이 팔로우하는 사람' 이 된다
following_query = QUERY_PREFIX + """
SELECT ?x WHERE {
  ex:민준 ex:팔로우 ?x .
}
"""
sparql_following = {str(row.x).split('/')[-1] for row in sns_graph.query(following_query)}
print('민준이 팔로우하는 사람:', sparql_following)

In [ ]:
# [자가채점]
assert sparql_following == {'서연', '지호', '하윤'}, \
    '주어를 ex:민준 으로 고정하고 목적어만 변수로 두었는지 확인하세요'
print('✅ 통과!')

**해설**: 3번의 파이썬 결과와 같습니다. 다른 점은 "어떻게 찾을지"를 적지 않고 **"무엇을 찾을지"** 만 적었다는 것입니다.

## 10. 타입으로 찾기
**배경**: 이 그래프에는 `ex:데일리카페 a ex:브랜드` 처럼 **타입**이 적혀 있습니다. `a` 는 `rdf:type` 의 약어입니다.

**요구사항**:
- 타입이 `ex:브랜드` 인 것의 **이름** 집합 **`brands`** 를 만드세요.

**예시**: 브랜드는 **2개**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 술어 자리에 a 를 쓰고 목적어를 브랜드로 고정한 뒤, 주어를 변수로 둔다.

세부구현:
1. WHERE 안에 주어를 변수로, 술어를 a 로, 목적어를 ex:브랜드 로 적는다.
2. 결과의 주어를 이름으로 바꿔 집합에 담는다.
```

</details>

In [ ]:
# a 는 rdf:type 의 약어다. '타입이 브랜드인 것' 을 찾는다
brand_query = QUERY_PREFIX + """
SELECT ?b WHERE {
  ?b a ex:브랜드 .
}
"""
brands = {str(row.b).split('/')[-1] for row in sns_graph.query(brand_query)}
print('브랜드:', brands)

In [ ]:
# [자가채점]
assert brands == {'데일리카페', '액티브짐'}, \
    '술어 자리에 a 를 쓰고 목적어를 ex:브랜드 로 고정했는지 확인하세요'
print('✅ 통과!')

**해설**: 6번에서 파이썬 dict 의 `type` 키로 세던 것을 RDF 는 **트리플 하나**로 적어 둡니다. 그래서 타입도 질의로 찾을 수 있습니다.

## 11. 값으로 거르기: 리터럴 FILTER
**배경**: 지역은 다른 개체를 가리키는 것이 아니라 `"부산"` 같은 **값(리터럴)** 입니다. 값 비교는 `FILTER` 로 합니다.

**요구사항**:
- 타입이 `ex:사용자` 이면서 `ex:지역` 값이 `"부산"` 인 사람의 **이름** 집합 **`busan_users`** 를 만드세요.

**예시**: 부산에 사는 사용자는 **3명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 타입 패턴과 지역 패턴을 같은 변수로 잇고, 지역 값에 FILTER 를 건다.

세부구현:
1. WHERE 안에 사용자 타입 패턴과 지역 패턴을 두 줄로 적고 주어에 같은 변수를 쓴다.
2. FILTER 로 지역 변수가 부산과 같은지 비교한다(값은 따옴표로 감싼다).
3. 결과의 주어를 이름으로 바꿔 집합에 담는다.
```

</details>

In [ ]:
# 지역은 리터럴이라 따옴표로 감싼 값과 비교한다
busan_query = QUERY_PREFIX + """
SELECT ?u WHERE {
  ?u a ex:사용자 .
  ?u ex:지역 ?region .
  FILTER(?region = "부산")
}
"""
busan_users = {str(row.u).split('/')[-1] for row in sns_graph.query(busan_query)}
print('부산 사용자:', busan_users)

In [ ]:
# [자가채점]
assert busan_users == {'수아', '지아', '지호'}, \
    '지역 값을 FILTER 로 비교했는지, 사용자 타입으로 좁혔는지 확인하세요'
print('✅ 통과!')

**해설**: 5번의 파이썬 필터와 같은 일입니다. 값(리터럴)은 `ex:부산` 처럼 주소로 쓰지 않고 **따옴표로 감싼 글자** 그대로 비교합니다.

## 12. 정렬하고 잘라 내기
**배경**: 결과가 많으면 **정렬**해서 앞부분만 봅니다. `ORDER BY` 로 줄을 세우고 `LIMIT` 으로 개수를 자릅니다. 같은 사람이 여러 번 나오지 않게 `DISTINCT` 도 씁니다.

**요구사항**:
- 누군가에게 **팔로우당하는 사람**(`ex:팔로우` 의 목적어)을 이름 순으로 정렬해 **앞 3명**의 이름을 **리스트** **`followed_top3`** 에 **순서대로** 담으세요.
- 집합이 아니라 **리스트**입니다(순서가 채점 대상입니다).

**예시**: 앞 3명은 **3개** 원소의 리스트입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 목적어를 변수로 둔 패턴 하나를 쓰고, 중복을 없앤 뒤 정렬해 앞에서 세 개만 자른다.

세부구현:
1. SELECT 뒤에 DISTINCT 를 붙이고 목적어 변수를 적는다.
2. ORDER BY 로 그 변수를 정렬하고 LIMIT 으로 3 개만 남긴다.
3. 결과를 순서대로 리스트에 담는다(집합으로 만들면 순서가 사라진다).
```

</details>

In [ ]:
# DISTINCT 로 중복을 없애고, ORDER BY 로 정렬한 뒤 LIMIT 으로 앞 3개만 남긴다
top3_query = QUERY_PREFIX + """
SELECT DISTINCT ?x WHERE {
  ?p ex:팔로우 ?x .
}
ORDER BY ?x
LIMIT 3
"""
followed_top3 = [str(row.x).split('/')[-1] for row in sns_graph.query(top3_query)]
print('팔로우당하는 사람 앞 3명:', followed_top3)

In [ ]:
# [자가채점]
assert followed_top3 == ['도윤', '민준', '서연'], \
    'DISTINCT·ORDER BY·LIMIT 을 모두 썼는지, 리스트로 순서를 지켰는지 확인하세요'
print('✅ 통과!')

**해설**: `ORDER BY` 를 빼면 순서가 실행할 때마다 달라질 수 있습니다. `LIMIT` 은 정렬 **뒤에** 적용되므로 둘은 늘 함께 씁니다.

## 13. 없어도 빼지 않기: OPTIONAL
**배경**: 관심 브랜드가 **없는** 사용자도 있습니다. 패턴을 그냥 적으면 그런 사람은 결과에서 사라집니다. `OPTIONAL` 을 쓰면 빈칸으로 남습니다.

**요구사항**:
- 사용자마다 관심 브랜드를 사전 **`interest_map`**(키 = 사용자 이름, 값 = 브랜드 이름)에 담으세요.
- 관심 브랜드가 없는 사용자는 값으로 문자열 **`'없음'`** 을 넣으세요.

**예시**: 사용자 **8명**이 모두 키로 들어가고, 그중 **4명**은 값이 `'없음'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 사용자 타입 패턴을 적고, 관심 패턴만 OPTIONAL 로 감싼다.

세부구현:
1. WHERE 안에 사용자 타입 패턴 한 줄을 적는다.
2. 그 아래에 OPTIONAL 을 적고 중괄호 안에 관심 패턴을 넣는다.
3. 결과를 돌며 브랜드 변수가 비어 있으면 없음 을, 아니면 이름을 사전에 담는다.
```

</details>

In [ ]:
# OPTIONAL 로 감싸면 관심 브랜드가 없어도 그 사용자 행이 남는다
interest_query = QUERY_PREFIX + """
SELECT ?u ?b WHERE {
  ?u a ex:사용자 .
  OPTIONAL { ?u ex:관심 ?b . }
}
"""
interest_map = {}
for row in sns_graph.query(interest_query):
    user = str(row.u).split('/')[-1]
    brand = '없음'
    if row.b:                      # 관심 브랜드가 없는 행은 None 이 온다
        brand = str(row.b).split('/')[-1]
    interest_map[user] = brand
print(interest_map)

In [ ]:
# [자가채점]
assert interest_map == {'도윤': '없음', '민준': '데일리카페', '서연': '없음', '수아': '없음', '예준': '액티브짐', '지아': '없음', '지호': '데일리카페', '하윤': '액티브짐'}, \
    '관심 패턴만 OPTIONAL 로 감쌌는지, 없는 사람에 없음 을 넣었는지 확인하세요'
print('✅ 통과!')

**해설**: `OPTIONAL` 을 빼면 관심 브랜드가 있는 4명만 남습니다. "있으면 보여 주고 없어도 빼지 마라"가 필요할 때 쓰는 문법입니다.

## 14. 있는지만 묻기: ASK
**배경**: 행이 필요 없고 "그런 사실이 있나"만 궁금할 때가 있습니다. `ASK` 는 참/거짓만 돌려줍니다.

**요구사항**:
- `민준`이 `액티브짐`에 관심 있는지 `ASK` 로 물어 참/거짓을 변수 **`minjun_gym`** 에 담으세요(`bool()` 로 감싸면 True/False 가 됩니다).
- 같은 방식으로 `민준`이 `데일리카페`에 관심 있는지를 **`minjun_cafe`** 에 담으세요.

**예시**: 둘 중 하나만 참입니다.

<details><summary>힌트</summary>

```text
접근방법:
- ASK 는 SELECT 없이 ASK 와 중괄호 패턴만 적는다.

세부구현:
1. QUERY_PREFIX 뒤에 ASK 와 중괄호를 적고 그 안에 확인할 트리플을 그대로 쓴다.
2. 질의 결과를 bool 로 감싸 변수에 담는다.
```

</details>

In [ ]:
# ASK 는 행이 아니라 참/거짓만 돌려준다
minjun_gym = bool(sns_graph.query(QUERY_PREFIX + 'ASK { ex:민준 ex:관심 ex:액티브짐 . }'))
minjun_cafe = bool(sns_graph.query(QUERY_PREFIX + 'ASK { ex:민준 ex:관심 ex:데일리카페 . }'))
print('민준-액티브짐:', minjun_gym)
print('민준-데일리카페:', minjun_cafe)

In [ ]:
# [자가채점]
assert minjun_gym is False, 'ASK 결과를 bool 로 바꿔 담았는지 확인하세요'
assert minjun_cafe is True, '민준의 관심 브랜드를 데이터에서 확인해 보세요'
print('✅ 통과!')

**해설**: `ASK` 는 결과 행을 만들지 않아 "존재 확인"에 가볍습니다. 같은 것을 `SELECT` 로 물으면 행이 있는지 세어 봐야 합니다.

## 15. 2홉을 한 줄로: 속성 경로
**배경**: "민준이 팔로우하는 사람들이 관심 있는 브랜드"는 **2홉**입니다. 패턴 두 줄 대신 술어를 슬래시로 이어 **한 줄**로 적을 수 있습니다.

**요구사항**:
- 속성 경로를 써서 그 브랜드들의 **이름** 집합 **`friend_brands`** 를 만드세요.
- 패턴은 **한 줄**이어야 합니다(`팔로우` 다음 `관심`).

**예시**: 브랜드 **2개**가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 주어를 민준으로 고정하고 두 술어를 슬래시로 이어 목적어만 변수로 둔다.

세부구현:
1. WHERE 안에 패턴 한 줄을 적되 술어 자리에 두 술어를 슬래시로 잇는다.
2. 결과의 목적어를 이름으로 바꿔 집합에 담는다.
```

</details>

In [ ]:
# 술어를 슬래시로 이으면 2홉을 한 줄로 적을 수 있다
path_query = QUERY_PREFIX + """
SELECT DISTINCT ?b WHERE {
  ex:민준 ex:팔로우/ex:관심 ?b .
}
"""
friend_brands = {str(row.b).split('/')[-1] for row in sns_graph.query(path_query)}
print('팔로우하는 사람들의 관심 브랜드:', friend_brands)

In [ ]:
# [자가채점]
assert friend_brands == {'데일리카페', '액티브짐'}, \
    '두 술어를 슬래시로 이어 한 줄로 적었는지 확인하세요'
print('✅ 통과!')

**해설**: 8번의 2홉 순회를 질의 한 줄로 바꾼 것입니다. 중간에 거쳐 간 사람은 결과에 남지 않습니다. 그 사람도 보고 싶으면 두 줄로 풀어 쓰면 됩니다.

## 16. 계층을 타고 올라가기
**배경**: 이 그래프에는 `ex:사용자 rdfs:subClassOf ex:계정` 처럼 **타입끼리의 상하 관계**가 적혀 있습니다. 그런데 `?x a ex:계정` 으로 찾으면 **한 건도 안 나옵니다.** 어떤 노드에도 `a ex:계정` 이라고 직접 적혀 있지 않기 때문입니다. 계층을 타고 올라가려면 경로에 `*` 를 붙입니다(`*` 는 "그 술어를 **0홉 이상** 따라가라"는 뜻이라 자기 자신도 포함합니다).

**요구사항**:
- 먼저 `?x a ex:계정` 만으로 찾아 결과 개수를 변수 **`direct_count`** 에 담으세요.
- 그다음 `a` 뒤에 `rdfs:subClassOf*` 를 이어 붙인 경로로 찾아, 계정에 해당하는 것의 **이름** 집합 **`accounts`** 를 만드세요.

**예시**: 직접 찾으면 **0건**이고, 계층을 타고 올라가면 **10개**(사용자 + 브랜드 전부)가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 질문을 경로 없이 한 번, 경로를 붙여 한 번 던져 결과를 견준다.

세부구현:
1. 술어를 a 로만 둔 질의를 만들어 실행하고 결과 개수를 센다.
2. 술어 자리에 a 와 상위 개념 술어를 슬래시로 잇고 뒤에 별표를 붙인다.
3. 두 번째 결과의 주어를 이름으로 바꿔 집합에 담는다.
```

</details>

In [ ]:
# 직접 적힌 것만 찾으면 한 건도 없다. 계정 이라고 적어 둔 노드가 없기 때문이다
direct_query = QUERY_PREFIX + 'SELECT ?x WHERE { ?x a ex:계정 . }'
direct_count = len(list(sns_graph.query(direct_query)))

# a 뒤에 상위 개념을 0홉 이상 따라가는 경로를 붙이면 계층을 타고 올라간다
account_query = QUERY_PREFIX + """
SELECT ?x WHERE {
  ?x a/rdfs:subClassOf* ex:계정 .
}
"""
accounts = {str(row.x).split('/')[-1] for row in sns_graph.query(account_query)}
print('직접 찾기:', direct_count)
print('계층 타고:', len(accounts), accounts)

In [ ]:
# [자가채점]
assert direct_count == 0, \
    '계정 이라고 직접 적힌 노드는 없습니다. 경로 없이 그대로 찾아 보세요'
assert accounts == {'데일리카페', '도윤', '민준', '서연', '수아', '액티브짐', '예준', '지아', '지호', '하윤'}, \
    'a 뒤에 rdfs:subClassOf* 를 이어 붙였는지 확인하세요(별표를 빼면 0건입니다)'
print('✅ 통과!')

**해설**: 이것이 실제 지식 그래프에서 **가장 자주 쓰이는 관용구**입니다. Wikidata 에서도 "이 사람의 직업이 물리학자 계열인가"를 물을 때 `wdt:P106/wdt:P279*` 처럼 똑같은 모양으로 씁니다. 교안 5절에서 이 경로를 붙였을 때 물리학자가 5만 9천 명에서 7만 2천 명으로 늘어난 것과 같은 원리입니다.

**흔한 실수**: 1) 별표를 빼서 0건이 나오는 경우(직접 적힌 것만 찾습니다). 2) `a` 없이 `rdfs:subClassOf*` 만 써서 타입 노드끼리만 찾는 경우.

---
수고했어요! LV1 에서 차수·관계 세기, 방향(팔로잉/팔로워), 속성 필터, 레이블 분류, 트리플 변환, 2홉 순회를 파이썬으로 익히고, 이어서 **SPARQL 문법을 하나씩** 써 봤습니다(기본 패턴·`a`·리터럴 `FILTER`·`ORDER BY`/`LIMIT`·`OPTIONAL`·`ASK`·속성 경로). LV2 에서는 이것들을 **조합**해 음악 그래프의 추천과 질의를 다룹니다.